In [ ]:
import os
import gc
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)


class CFG:
    seed = 42
    target = "PitNextLap"
    id_col = "id"
    comp_paths = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
    ]
    original_paths = [
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
    ]


def seed_everything(seed: int = 42):
    import random

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def first_existing_path(paths):
    for path in paths:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"No valid path found from: {paths}")


def print_section(title: str):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


seed_everything(CFG.seed)


### LoadData

In [ ]:
print_section("Loading Data")

COMP_PATH = first_existing_path(CFG.comp_paths)

train_raw = pd.read_csv(os.path.join(COMP_PATH, "train.csv"))
test_raw = pd.read_csv(os.path.join(COMP_PATH, "test.csv"))
sample_submission = pd.read_csv(os.path.join(COMP_PATH, "sample_submission.csv"))

print(f"Train shape: {train_raw.shape}")
print(f"Test shape : {test_raw.shape}")

if CFG.target in train_raw.columns:
    print(f"Train target rate: {train_raw[CFG.target].mean():.6f}")
else:
    print("WARNING: target column not found in train")

original_raw = None
for path in CFG.original_paths:
    if os.path.exists(path):
        original_raw = pd.read_csv(path)
        print(f"Loaded original data from: {path}, shape = {original_raw.shape}")
        break

if original_raw is None:
    print("Original data NOT found (this is ok but we lose extra context).")
elif CFG.target in original_raw.columns:
    print(f"Original target rate: {original_raw[CFG.target].mean():.6f}")
else:
    print("Original data has no target column — check schema.")

print_section("Column Overview: Competition Train")
print(train_raw.dtypes)

print_section("Column Overview: Original (if available)")
if original_raw is not None:
    print(original_raw.dtypes)

gc.collect()


In [ ]:
print_section("Basic EDA: Schema and Missingness")


def summarize_schema(df, name):
    print(f"\n{name} schema:")
    info = []

    for col in df.columns:
        s = df[col]
        info.append(
            {
                "column": col,
                "dtype": s.dtype,
                "missing": s.isna().sum(),
                "missing_pct": 100 * s.isna().mean(),
                "n_unique": s.nunique(),
            }
        )

    info_df = pd.DataFrame(info)
    print(info_df.sort_values("missing_pct", ascending=False).to_string(index=False))
    return info_df


train_schema = summarize_schema(train_raw, "Train")
test_schema = summarize_schema(test_raw, "Test")
original_schema = summarize_schema(original_raw, "Original") if original_raw is not None else None

print_section("Duplicate and ID checks")


def duplicate_report(df, name, id_col):
    n_rows = len(df)
    dup_ids = df[id_col].duplicated().sum() if id_col in df.columns else "N/A"
    dup_full = df.duplicated().sum()
    print(f"{name}: rows={n_rows}, dup_ids={dup_ids}, dup_full_rows={dup_full}")


duplicate_report(train_raw, "Train", CFG.id_col)
duplicate_report(test_raw, "Test", CFG.id_col)

if original_raw is not None and CFG.id_col in original_raw.columns:
    duplicate_report(original_raw, "Original", CFG.id_col)

NUMERIC_FEATURES = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]
CATEGORICAL_FEATURES = ["Driver", "Compound", "Race"]

print_section("Feature inventory")
print("Numeric features:", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)

feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
missing_train = train_raw[feature_cols].isna().sum().sum()
missing_test = test_raw[feature_cols].isna().sum().sum()

print(f"Total missing in train (model features): {missing_train}")
print(f"Total missing in test  (model features): {missing_test}")

if original_raw is not None:
    miss_orig = original_raw[feature_cols].isna().sum().sum()
    print(f"Total missing in original (model features): {miss_orig}")


In [ ]:
print_section("Numeric train/test drift (means and SMD)")


def numeric_drift(train_df, test_df, cols):
    rows = []

    for col in cols:
        a = train_df[col].astype(float)
        b = test_df[col].astype(float)
        mu_a, mu_b = a.mean(), b.mean()
        s = np.sqrt((a.var() + b.var()) / 2.0)
        smd = 0.0 if s == 0 else (mu_b - mu_a) / s
        rows.append(
            {
                "feature": col,
                "train_mean": mu_a,
                "test_mean": mu_b,
                "SMD": smd,
            }
        )

    drift_df = pd.DataFrame(rows)
    print(drift_df.sort_values("SMD", key=np.abs).to_string(index=False))
    return drift_df


drift_df = numeric_drift(train_raw, test_raw, NUMERIC_FEATURES)

print_section("Categorical train/test coverage")


def cat_coverage(train_df, test_df, cols):
    rows = []

    for col in cols:
        tr_vals = set(train_df[col].astype("category").unique())
        te_vals = set(test_df[col].astype("category").unique())
        unseen_in_test = te_vals - tr_vals
        rows.append(
            {
                "feature": col,
                "train_unique": len(tr_vals),
                "test_unique": len(te_vals),
                "unseen_in_test": len(unseen_in_test),
            }
        )

    cov_df = pd.DataFrame(rows)
    print(cov_df.to_string(index=False))
    return cov_df


cov_df = cat_coverage(train_raw, test_raw, CATEGORICAL_FEATURES)


In [ ]:
print_section("Feature Engineering: domain features")

train_fe = train_raw.copy()
test_fe = test_raw.copy()
original_fe = original_raw.copy() if original_raw is not None and CFG.target in original_raw.columns else None

train_fe["IsOriginalData"] = 0
test_fe["IsOriginalData"] = 0

if original_fe is not None:
    original_fe["IsOriginalData"] = 1

BASE_CAT_COLS = ["Driver", "Compound", "Race"]


def safe_div(a, b, eps=1e-6):
    return a / (b + eps)


def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    for col in BASE_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    has = lambda cols: set(cols).issubset(out.columns)

    if has(["LapNumber", "RaceProgress"]):
        rp = out["RaceProgress"].clip(lower=eps)
        est_total = safe_div(out["LapNumber"], rp, eps).replace([np.inf, -np.inf], np.nan)
        out["EstimatedTotalLaps"] = est_total.clip(1, 120)
        out["LapsRemaining"] = (out["EstimatedTotalLaps"] - out["LapNumber"]).clip(lower=0)
        out["RemainingRaceProgress"] = 1.0 - out["RaceProgress"]
        out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]
        out["RacePhase"] = pd.cut(
            out["RaceProgress"],
            bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
            labels=["P1", "P2", "P3", "P4", "P5"],
        ).astype(str)
        out["LapBin"] = pd.cut(
            out["LapNumber"],
            bins=[-np.inf, 5, 10, 20, 35, 50, np.inf],
            labels=["L_000_005", "L_006_010", "L_011_020", "L_021_035", "L_036_050", "L_051_plus"],
        ).astype(str)

    if has(["TyreLife", "LapNumber"]):
        ln = out["LapNumber"].clip(lower=1)
        tl = out["TyreLife"].clip(lower=1)
        out["TyreAgeRatio"] = safe_div(out["TyreLife"], ln, eps)
        out["LapPerTyreLife"] = safe_div(out["LapNumber"], out["TyreLife"] + 1, eps)
        out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

    if has(["TyreLife", "EstimatedTotalLaps"]):
        out["TyreAgeVsRace"] = safe_div(out["TyreLife"], out["EstimatedTotalLaps"].clip(lower=1), eps)

    if has(["TyreLife", "RaceProgress"]):
        out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]

    if has(["Stint", "TyreLife"]):
        out["StintPressure"] = out["Stint"] * out["TyreLife"]
        out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]
        out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

    if "TyreLife" in out.columns:
        out["TyreLifeBin"] = pd.cut(
            out["TyreLife"],
            bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
            labels=["T_000_003", "T_004_007", "T_008_012", "T_013_020", "T_021_030", "T_031_plus"],
        ).astype(str)

    if "Position" in out.columns:
        out["PositionBin"] = pd.cut(
            out["Position"],
            bins=[-np.inf, 3, 8, 14, np.inf],
            labels=["front", "upper_mid", "lower_mid", "back"],
        ).astype(str)

    if has(["Cumulative_Degradation", "LapNumber"]):
        out["DegPerRaceLap"] = safe_div(
            out["Cumulative_Degradation"],
            out["LapNumber"].clip(lower=1),
            eps,
        )

    if has(["Cumulative_Degradation", "TyreLife"]):
        out["DegPerTyreLap"] = safe_div(
            out["Cumulative_Degradation"],
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if "LapTime_Delta" in out.columns:
        out["DeltaAbs"] = out["LapTime_Delta"].abs()

    if "Position_Change" in out.columns:
        out["Abs_Position_Change"] = out["Position_Change"].abs()
        out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
        out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

    if has(["Position", "RaceProgress"]):
        out["PositionPressure"] = out["Position"] * out["RaceProgress"]

    def make_cross(name, cols):
        if set(cols).issubset(out.columns):
            val = out[cols[0]].astype(str)
            for col in cols[1:]:
                val = val + "_" + out[col].astype(str)
            out[name] = val

    make_cross("Race_Year", ["Race", "Year"])
    make_cross("Compound_Stint", ["Compound", "Stint"])
    make_cross("Race_Compound", ["Race", "Compound"])
    make_cross("RacePhase_TyreLifeBin", ["RacePhase", "TyreLifeBin"])

    out = out.replace([np.inf, -np.inf], np.nan)
    float_cols = out.select_dtypes(include=["float64"]).columns

    for col in float_cols:
        out[col] = out[col].astype(np.float32)

    return out


train_fe = add_domain_features(train_fe)
test_fe = add_domain_features(test_fe)

if original_fe is not None:
    if "Normalized_TyreLife" in original_fe.columns:
        original_fe = original_fe.drop(columns=["Normalized_TyreLife"])
    original_fe = add_domain_features(original_fe)

print(f"Train FE shape: {train_fe.shape}")
print(f"Test FE shape : {test_fe.shape}")

if original_fe is not None:
    print(f"Original FE shape: {original_fe.shape}")


In [ ]:
print_section("Align columns between train/test/original")

EXCLUDE_COLS = [CFG.id_col, CFG.target]


def align_columns(train_df, test_df, original_df=None):
    train = train_df.copy()
    test = test_df.copy()
    orig = original_df.copy() if original_df is not None else None

    common_feats = [c for c in train.columns if c in test.columns and c not in EXCLUDE_COLS]
    train = train[common_feats + ([CFG.target] if CFG.target in train.columns else [])]
    test = test[common_feats]

    if orig is not None and CFG.target in orig.columns:
        for col in common_feats:
            if col not in orig.columns:
                orig[col] = np.nan
        orig = orig[common_feats + [CFG.target]]
    else:
        orig = None

    print(f"Aligned train shape   : {train.shape}")
    print(f"Aligned test  shape   : {test.shape}")

    if orig is not None:
        print(f"Aligned original shape: {orig.shape}")

    return train, test, orig, common_feats


train_fe, test_fe, original_fe, FEATURE_COLS = align_columns(train_fe, test_fe, original_fe)

print_section("Fill missing values and detect categorical features")


def fill_missing_and_get_types(train_df, test_df, original_df=None):
    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    all_feature_cols = [c for c in train_df.columns if c != CFG.target]
    cat_cols = []

    for col in all_feature_cols:
        is_cat = False
        for frame in frames:
            if col not in frame.columns:
                continue

            dtype = frame[col].dtype
            if dtype == "object" or str(dtype).startswith("category") or str(dtype).startswith("string"):
                is_cat = True
                break

        if is_cat:
            cat_cols.append(col)

    num_cols = [c for c in all_feature_cols if c not in cat_cols]

    for col in cat_cols:
        values = pd.concat(
            [frame[col].astype("string") for frame in frames if col in frame.columns],
            axis=0,
        )
        mode_val = values.mode().iloc[0] if len(values.mode()) else "__MISSING__"

        for frame in frames:
            if col in frame.columns:
                frame[col] = frame[col].astype("string").fillna(mode_val).astype(str)

    for col in num_cols:
        values = pd.concat(
            [frame[col] for frame in frames if col in frame.columns],
            axis=0,
        )
        fill_val = values.replace([np.inf, -np.inf], np.nan).median()

        for frame in frames:
            if col in frame.columns:
                frame[col] = frame[col].replace([np.inf, -np.inf], np.nan).fillna(fill_val)
                if frame[col].dtype == "float64":
                    frame[col] = frame[col].astype(np.float32)

    print(f"Total categorical features: {len(cat_cols)}")
    print(f"Total numeric features     : {len(num_cols)}")
    return train_df, test_df, original_df, cat_cols, num_cols


train_fe, test_fe, original_fe, CAT_COLS, NUM_COLS = fill_missing_and_get_types(
    train_fe,
    test_fe,
    original_fe,
)

print(f"Final train FE shape: {train_fe.shape}")
print(f"Final test  FE shape: {test_fe.shape}")

if original_fe is not None:
    print(f"Final original FE shape: {original_fe.shape}")


In [ ]:
print_section("Preparing matrices for CatBoost")

X_comp = train_fe.drop(columns=[CFG.target], errors="ignore")
y_comp = train_fe[CFG.target].astype(int)
X_test = test_fe.copy()

if original_fe is not None and CFG.target in original_fe.columns:
    X_orig = original_fe.drop(columns=[CFG.target], errors="ignore")
    y_orig = original_fe[CFG.target].astype(int)
else:
    X_orig, y_orig = None, None

common_features = [c for c in X_comp.columns if c in X_test.columns]
X_comp = X_comp[common_features]
X_test = X_test[common_features]

if X_orig is not None:
    for col in common_features:
        if col not in X_orig.columns:
            X_orig[col] = np.nan
    X_orig = X_orig[common_features]

print(f"X_comp shape: {X_comp.shape}")
print(f"X_test shape: {X_test.shape}")

if X_orig is not None:
    print(f"X_orig shape: {X_orig.shape}")

print(f"Target mean (competition): {y_comp.mean():.6f}")

if y_orig is not None:
    print(f"Target mean (original)   : {y_orig.mean():.6f}")

CAT_COLS_FINAL = [c for c in CAT_COLS if c in common_features]
NUM_COLS_FINAL = [c for c in common_features if c not in CAT_COLS_FINAL]
cat_features_idx = [X_comp.columns.get_loc(c) for c in CAT_COLS_FINAL]

print_section("Final feature groups")
print(f"Total features      : {len(common_features)}")
print(f"Categorical features: {len(CAT_COLS_FINAL)}")
print(f"Numeric features    : {len(NUM_COLS_FINAL)}")
print("Some categorical features:", CAT_COLS_FINAL[:10])


In [ ]:
print_section("Stratified KFold validation with CatBoost")

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, log_loss, f1_score, precision_score, recall_score
from catboost import CatBoostClassifier

N_FOLDS = 5
groups = train_raw["Race"].astype(str) + "_" + train_raw["Year"].astype(str)
skf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=CFG.seed,
)

oof_pred = np.zeros(len(X_comp), dtype=float)
best_iters = []
fold_metrics = []


def get_catboost_params(seed, iterations):
    return {
        "iterations": iterations,
        "learning_rate": 0.018,
        "depth": 8,
        "l2_leaf_reg": 8.5,
        "random_strength": 0.65,
        "bootstrap_type": "Bayesian",
        "bagging_temperature": 0.45,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "auto_class_weights": "Balanced",
        "task_type": "GPU",
        "devices": "0:1",
        "random_seed": seed,
        "early_stopping_rounds": 500,
        "allow_writing_files": False,
        "verbose": 200,
    }


for fold, (tr_idx, val_idx) in enumerate(
    skf.split(X_comp, y_comp, groups=groups),
    1,
):
    print_section(f"Fold {fold}/{N_FOLDS}")

    X_tr_comp = X_comp.iloc[tr_idx].reset_index(drop=True)
    y_tr_comp = y_comp.iloc[tr_idx].reset_index(drop=True)
    X_val = X_comp.iloc[val_idx].reset_index(drop=True)
    y_val = y_comp.iloc[val_idx].reset_index(drop=True)

    if X_orig is not None and y_orig is not None:
        X_tr = pd.concat(
            [X_tr_comp, X_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat(
            [y_tr_comp, y_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
    else:
        X_tr = X_tr_comp.copy()
        y_tr = y_tr_comp.copy()

    print(f"Train fold shape: {X_tr.shape}, Valid shape: {X_val.shape}")

    params = get_catboost_params(seed=CFG.seed + fold, iterations=11000)
    model = CatBoostClassifier(**params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=(X_val, y_val),
        cat_features=cat_features_idx,
        use_best_model=True,
    )

    val_pred = model.predict_proba(X_val)[:, 1]
    val_pred = np.clip(val_pred, 1e-7, 1 - 1e-7)
    oof_pred[val_idx] = val_pred

    best_iter = model.get_best_iteration()
    best_iters.append(best_iter)

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_logloss = log_loss(y_val, val_pred)

    thresholds = np.linspace(0.05, 0.95, 181)
    best_thr = 0.5
    best_f1 = -1.0
    for threshold in thresholds:
        y_hat = (val_pred >= threshold).astype(int)
        f1 = f1_score(y_val, y_hat)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = threshold

    y_hat_05 = (val_pred >= 0.5).astype(int)
    y_hat_best = (val_pred >= best_thr).astype(int)

    metrics_fold = {
        "fold": fold,
        "auc": fold_auc,
        "logloss": fold_logloss,
        "f1_05": f1_score(y_val, y_hat_05),
        "precision_05": precision_score(y_val, y_hat_05),
        "recall_05": recall_score(y_val, y_hat_05),
        "best_thr": best_thr,
        "f1_best": f1_score(y_val, y_hat_best),
        "precision_best": precision_score(y_val, y_hat_best),
        "recall_best": recall_score(y_val, y_hat_best),
        "best_iter": best_iter,
    }
    fold_metrics.append(metrics_fold)

    print(f"Fold {fold} AUC       : {fold_auc:.6f}")
    print(f"Fold {fold} LogLoss   : {fold_logloss:.6f}")
    print(f"Fold {fold} best_iter : {best_iter}")
    print(f"Fold {fold} F1@0.5    : {metrics_fold['f1_05']:.6f}")
    print(
        f"Fold {fold} F1@best   : {metrics_fold['f1_best']:.6f} "
        f"(thr={best_thr:.4f})"
    )

    gc.collect()

print_section("OOF metrics (StratifiedGroupKFold)")
oof_auc = roc_auc_score(y_comp, oof_pred)
oof_logloss = log_loss(y_comp, oof_pred)

thresholds = np.linspace(0.05, 0.95, 181)
best_thr_global = 0.5
best_f1_global = -1.0
for threshold in thresholds:
    y_hat = (oof_pred >= threshold).astype(int)
    f1 = f1_score(y_comp, y_hat)
    if f1 > best_f1_global:
        best_f1_global = f1
        best_thr_global = threshold

y_hat_05 = (oof_pred >= 0.5).astype(int)
y_hat_best = (oof_pred >= best_thr_global).astype(int)

print(f"OOF AUC        : {oof_auc:.6f}")
print(f"OOF LogLoss    : {oof_logloss:.6f}")
print(f"OOF F1@0.5     : {f1_score(y_comp, y_hat_05):.6f}")
print(
    f"OOF F1@best    : {f1_score(y_comp, y_hat_best):.6f} "
    f"(thr={best_thr_global:.4f})"
)
print(f"OOF Precision@best: {precision_score(y_comp, y_hat_best):.6f}")
print(f"OOF Recall@best   : {recall_score(y_comp, y_hat_best):.6f}")

best_iters_arr = np.array(best_iters)
print(f"Best iterations per fold: {best_iters_arr}")
print(f"Mean(best_iter): {best_iters_arr.mean():.1f}")

In [ ]:
print_section("Final full-data CatBoost model and submission")

if "best_iters" not in globals() or len(best_iters) == 0:
    raise RuntimeError("best_iters is empty — убедись, что KFold ячейка была запущена.")

best_iters_arr = np.array(best_iters)
mean_best_iter = best_iters_arr.mean()
final_iterations = int(mean_best_iter * 1.10)
final_iterations = max(1800, final_iterations)

print(f"Best iterations per fold: {best_iters_arr}")
print(f"Mean(best_iter)         : {mean_best_iter:.1f}")
print(f"Final model iterations  : {final_iterations}")

if X_orig is not None and y_orig is not None:
    X_full = pd.concat(
        [X_comp.reset_index(drop=True), X_orig.reset_index(drop=True)],
        axis=0,
        ignore_index=True,
    )
    y_full = pd.concat(
        [y_comp.reset_index(drop=True), y_orig.reset_index(drop=True)],
        axis=0,
        ignore_index=True,
    )
else:
    X_full = X_comp.copy()
    y_full = y_comp.copy()

print(f"Full training shape: {X_full.shape}")
print(f"Test shape         : {X_test.shape}")

final_params = {
    "iterations": final_iterations,
    "learning_rate": 0.018,
    "depth": 8,
    "l2_leaf_reg": 8.5,
    "random_strength": 0.65,
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 0.45,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
    "task_type": "GPU",
    "devices": "0:1",
    "random_seed": CFG.seed + 999,
    "allow_writing_files": False,
    "verbose": 300,
}

final_model = CatBoostClassifier(**final_params)
final_model.fit(
    X_full,
    y_full,
    cat_features=cat_features_idx,
)

test_pred = final_model.predict_proba(X_test)[:, 1]
test_pred = np.clip(test_pred, 1e-7, 1 - 1e-7)

print_section("Test prediction distribution")
print(f"Min   : {test_pred.min():.6f}")
print(f"1%    : {np.percentile(test_pred, 1):.6f}")
print(f"5%    : {np.percentile(test_pred, 5):.6f}")
print(f"25%   : {np.percentile(test_pred, 25):.6f}")
print(f"Median: {np.median(test_pred):.6f}")
print(f"75%   : {np.percentile(test_pred, 75):.6f}")
print(f"95%   : {np.percentile(test_pred, 95):.6f}")
print(f"99%   : {np.percentile(test_pred, 99):.6f}")
print(f"Max   : {test_pred.max():.6f}")
print(f"Mean  : {test_pred.mean():.6f}")

final_fi = pd.DataFrame(
    {
        "feature": X_full.columns,
        "importance": final_model.get_feature_importance(),
    }
).sort_values("importance", ascending=False)

print_section("Top final feature importance")
print(final_fi.head(40).to_string(index=False))

print_section("Saving submission.csv")

submission = sample_submission.copy()
target_col = CFG.target if CFG.target in submission.columns else [c for c in submission.columns if c != CFG.id_col][0]
submission[target_col] = test_pred
submission.to_csv("submission.csv", index=False)

oof_df = pd.DataFrame(
    {
        "y_true": y_comp.values,
        "oof_pred": oof_pred,
    }
)
oof_df.to_csv("oof_predictions.csv", index=False)
final_fi.to_csv("final_feature_importance.csv", index=False)

print("Saved submission.csv")
print("Saved oof_predictions.csv")
print("Saved final_feature_importance.csv")

print_section("Finished")
